## 最終課題

In [15]:
#1初期設定
from collections import deque
import requests
from bs4 import BeautifulSoup
import time
import urllib.parse

#トップページURL
url = "https://www.musashino-u.ac.jp"

#レスポンスの確認
res = requests.get(url)
res.encoding = res.apparent_encoding
print(f"ステータスコード:{res.status_code}")
print(f"ステータスメッセージ:{res.reason}")

visited_urls = set()  # 探索済みURLの集合
url_queue = deque([url])  # 未探索URLのリスト
url_title_dict = {}  # URLとタイトルを格納する辞書


ステータスコード:200
ステータスメッセージ:OK


In [16]:
# 2 クロール処理

#除外するファイル拡張子
excluded_extensions = ['.pdf', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx', '.zip', '.jpg', '.jpeg', '.png', '.gif', '.mp3', '.mp4', '.avi', '.mov']

while url_queue:
    current_url = url_queue.popleft()  # 未探索URLのリストから次のURLを取得する
    
    if current_url in visited_urls:
        continue  # 既に探索済みならスキップしてループを防ぐ
    
    visited_urls.add(current_url)  
    
    try:
        print(f"アクセス中: {current_url}")
        response = requests.get(current_url, timeout=10)
        response.encoding = response.apparent_encoding  # 文字化け防止
        
        time.sleep(0.5)  # サーバー負荷を避ける
        
        if response.status_code == 200:
            content_type = response.headers.get('Content-Type', '').lower()
            
            if 'text/html' not in content_type and 'application/xhtml+xml' not in content_type:
                continue  
            
            soup = BeautifulSoup(response.text, 'html.parser')  # HTMLを解析
            
            # タイトルを保存
            title = soup.find('title')
            if title and title.string:
                url_title_dict[current_url] = title.string.strip()
            else:
                url_title_dict[current_url] = "[タイトルなし]"
            
            # ページ内リンクを収集
            links = [a.get('href') for a in soup.find_all('a') if a.get('href')]
            
            for link in links:
                if any(link.lower().endswith(ext) for ext in excluded_extensions):
                    continue
                
                absolute_url = urllib.parse.urljoin(current_url, link)
                absolute_url = absolute_url.split('#')[0]
                
                parsed_url = urllib.parse.urlparse(absolute_url)
                parsed_base = urllib.parse.urlparse(url)
                is_same_domain = parsed_url.netloc == parsed_base.netloc or not parsed_url.netloc
                
                if (is_same_domain and 
                    absolute_url not in visited_urls and 
                    absolute_url not in url_queue):
                    url_queue.append(absolute_url)
        else:
            print(f"ステータスコード: {response.status_code} for {current_url}")
                    
    except requests.exceptions.RequestException as e:
        print(f"アクセスエラー {current_url}: {e}")

print("収集完了")
print(f"取得したページ数: {len(url_title_dict)}")

アクセス中: https://www.musashino-u.ac.jp
アクセス中: https://www.musashino-u.ac.jp/
アクセス中: https://www.musashino-u.ac.jp/access.html
アクセス中: https://www.musashino-u.ac.jp/admission/request.html
アクセス中: https://www.musashino-u.ac.jp/contact.html
アクセス中: https://www.musashino-u.ac.jp/prospective-students.html
アクセス中: https://www.musashino-u.ac.jp/students.html
アクセス中: https://www.musashino-u.ac.jp/alumni.html
アクセス中: https://www.musashino-u.ac.jp/parents.html
アクセス中: https://www.musashino-u.ac.jp/business.html
アクセス中: https://www.musashino-u.ac.jp/guide/
アクセス中: https://www.musashino-u.ac.jp/guide/profile/
アクセス中: https://www.musashino-u.ac.jp/guide/activities/
アクセス中: https://www.musashino-u.ac.jp/guide/campus/
アクセス中: https://www.musashino-u.ac.jp/guide/facility/
アクセス中: https://www.musashino-u.ac.jp/guide/information/
アクセス中: https://www.musashino-u.ac.jp/guide/profile/media/
アクセス中: https://www.musashino-u.ac.jp/admission/
アクセス中: https://www.musashino-u.ac.jp/admission/faculty/
アクセス中: https://www.musashino-

In [19]:
# 3 結果表示
# 格納された辞書型変数をprint文で表示
print("サイトマップ結果(ページのURL:<title>)")
print(url_title_dict)

サイトマップ結果(ページのURL:<title>)
{'https://www.musashino-u.ac.jp': '武蔵野大学', 'https://www.musashino-u.ac.jp/': '武蔵野大学', 'https://www.musashino-u.ac.jp/access.html': '交通アクセス | 武蔵野大学', 'https://www.musashino-u.ac.jp/admission/request.html': '資料請求 | 入試情報 | 武蔵野大学', 'https://www.musashino-u.ac.jp/contact.html': 'お問い合わせ | 武蔵野大学', 'https://www.musashino-u.ac.jp/prospective-students.html': '武蔵野大学で学びたい方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/students.html': '在学生の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/alumni.html': '卒業生の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/parents.html': '保護者の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/business.html': '企業・研究者の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/': '大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/profile/': '大学紹介 | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/activities/': '大学の取り組み | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/campus/': 'キャンパス | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/facility/': '附置機関・センター・附属施設 | 大学